In [ ]:
from datetime import datetime
import matplotlib.pyplot as plt
import os
import pandas as pd

from gs2d import load_image
from metrics import edgeness, shannon_entropy
from train import train

## Analysis settings

In [ ]:
dataset_dir = "data" # Directory containing image datasets as subdirectories
out_dir = f"output/analyze_{datetime.now().strftime("%Y%m%d_%H%M%S")}" # Analysis output directory
n_gaussians_range = range(25,201,25) # Range of numbers of Gaussians to test
out_width = 100 # Domain width
epochs = 100 # Training iterations

## Analyze dataset complexity

In [ ]:
# Measure entropy and edgeness for each image in each dataset
dataset_complexity = {'dataset': [], 'image': [], 'height': [], 'width': [], 
                      'shannon_entropy': [], 'edgeness': []}
for ds_name in sorted(os.listdir(dataset_dir)):
    if ds_name.startswith('.'):
        continue
    for img_basename in sorted(os.listdir(f"{dataset_dir}/{ds_name}")):
        if img_basename.startswith('.'):
            continue
        img = load_image(f"{dataset_dir}/{ds_name}/{img_basename}", width=out_width)
        dataset_complexity['dataset'].append(ds_name)
        dataset_complexity['image'].append(img_basename)
        dataset_complexity['height'].append(img.shape[0])
        dataset_complexity['width'].append(img.shape[1])
        dataset_complexity['shannon_entropy'].append(shannon_entropy(img))
        dataset_complexity['edgeness'].append(edgeness(img))

# Save complexity metrics
os.makedirs(out_dir, exist_ok=True)
dataset_complexity_df = pd.DataFrame(dataset_complexity)
dataset_complexity_df.to_csv(f"{out_dir}/dataset_complexity.csv")

In [ ]:
# Load complexity metrics
dataset_complexity_df = pd.read_csv(f"{out_dir}/dataset_complexity.csv")

# Gather metrics for plotting
ds_names = dataset_complexity_df['dataset'].unique()
entropy = []
edge = []
combined = []
for i, ds_name in enumerate(ds_names):
    rows = dataset_complexity_df['dataset'] == ds_name
    entropy.append(dataset_complexity_df[rows]['shannon_entropy'])
    edge.append(dataset_complexity_df[rows]['edgeness'])
    combined.append(entropy[-1] + edge[-1])

# Plot stacked histogram
fig, ax = plt.subplots()
ax.hist(combined, bins=7, histtype='barstacked', label=list(ds_names))
ax.legend(title="Dataset")
ax.set_xlabel('Image Complexity (Entropy + Edgeness)')
ax.set_ylabel('Count')
plt.show()
fig.savefig(f"{out_dir}/dataset_complexity.png")
plt.close(fig)

## Train 2DGS at varying `n_gaussians`

In [ ]:
# Measure 2DGS reconstruction quality for each image and n_gaussians
train_results = {'dataset': [], 'image': [], 'height': [], 'width': [], 
                 'shannon_entropy': [], 'edgeness': [],  
                 'n_gaussians': [], 'psnr': [], 'ssim': []}

# Iterate over each image and each number of Gaussians
for ds_name in sorted(os.listdir(dataset_dir)):
    if ds_name.startswith('.'):
        continue

    for target_basename in sorted(os.listdir(f"{dataset_dir}/{ds_name}")):
        if target_basename.startswith('.'):
            continue
        
        # Fit image with each number of Gaussians
        target_path = f"{dataset_dir}/{ds_name}/{target_basename}"
        train_out_dir = f"{out_dir}/train/{os.path.splitext(target_basename)[0]}"
        results_dict = {'n_gaussians': [], 'psnr': [], 'ssim': []}
        for n_gaussians in n_gaussians_range:
            with plt.ioff():
                train_stats = train(target_path, n_gaussians=n_gaussians, out_width=out_width, epochs=epochs, out_dir=train_out_dir)
            train_results['dataset'].append(ds_name)
            train_results['image'].append(target_basename)
            train_results['height'].append(train_stats['output_shape'][0])
            train_results['width'].append(train_stats['output_shape'][1])
            train_results['shannon_entropy'].append(train_stats['target_shannon_entropy'])
            train_results['edgeness'].append(train_stats['target_edgeness'])
            train_results['n_gaussians'].append(n_gaussians)
            train_results['psnr'].append(train_stats['psnr_final'])
            train_results['ssim'].append(train_stats['ssim_final'])
    
        # Update and save results
        train_results_df = pd.DataFrame(train_results)
        train_results_df.to_csv(f"{out_dir}/train_results.csv")

In [ ]:
# Load results JSON
train_results_df = pd.read_csv(f"{out_dir}/train_results.csv")

# Analyze
for n_gaussians in pd.unique(train_results_df['n_gaussians']):
    rows = train_results_df['n_gaussians'] == n_gaussians
    entropy = train_results_df['shannon_entropy'][rows]
    edge = train_results_df['edgeness'][rows]
    complexity = entropy + edge
    plt.scatter(complexity, train_results_df['psnr'][rows], label=n_gaussians)
plt.xlabel('Image Complexity (Entropy + Edgeness)')
plt.ylabel('PSNR')
plt.legend(title='n_gaussians')
plt.savefig(f"{out_dir}/analysis.png")
plt.show()
plt.close()